# **Notebook 3: Model Evaluation (Proof of Concept)**

>### **Project: PestNeuroVision**
>
>**Description:** This document describes the complete process of evaluating the model on the test dataset.
>
>**Important note about the dataset:** For this public repository and to demonstrate reproducibility, this notebook uses a selected subset of the data (miniset data). This subset consists exclusively of images licensed under CC0 1.0, in order to ensure full compliance with open access standards and intellectual property rights. The complete dataset used in the original research is not included because several of its photographs are subject to restrictive copyrights that prevent their public redistribution.
>
>The main objective of this notebook is to demonstrate the technical feasibility and functional integrity of the model evaluation. Although the complete results of the study presented in the article may include a larger, private, or restricted dataset, this script provides the exact environment and logic used during the research.
>
>---
>
>**License:** This notebook is licensed under the GNU Affero General Public License v3.0 (AGPL-3.0).
See details in the [GitHub repository](https://github.com/alexander-lm/PestNeuroVision/tree/main).

In [ ]:
!pip install Ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.6 MB/s eta 0:00:00


In [ ]:
import glob
import shutil
import os
import pandas as pd
import matplotlib
from IPython.display import Image, display
from google.colab import files
from ultralytics import settings, YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
if os.path.exists("trained_model"):
    shutil.rmtree("trained_model")
    print("Folder trained_model deleted.")

if os.path.exists("original_training_dataset"):
    shutil.rmtree("original_training_dataset")
    print("Folder original_training_dataset deleted.")


# Download the trained model (PesNeuroVision) and the test dataset from GitHub
!git init PestNeuroVision
%cd PestNeuroVision
!git remote add origin https://github.com/alexander-lm/PestNeuroVision.git
!git sparse-checkout init --cone
!git sparse-checkout set model-results/weights "dataset/mini_dataset_(CC0 1.0)"
!git pull --depth 1 origin main
!mv model-results/weights /content/weights
!mv "dataset/mini_dataset_(CC0 1.0)" /content/dataset
%cd /content
!rm -rf PestNeuroVision

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/PestNeuroVision/.git/
/content/PestNeuroVision
remote: Enumerating objects: 2004, done.
remote: Counting objects: 100% (2004/2004), done.
remote: Compressing objects: 100% (1995/1995), done.
remote: Total 2004 (delta 2), reused 1989 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (2004/2004), 245.61 MiB | 23.08 MiB/s, done.
Resolving deltas: 100% (2/2), done.
From https://github.com/alexander-lm/PestNeuroVision
 * branch            main       -> FETCH_HEAD

In [ ]:
import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  data = {
      'path': '/content/dataset',
      'train': 'train/images',
      'val': 'validation/images',
      'test': 'test/images',
      'nc': number_of_classes,
      'names': classes
  }

  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

path_to_classes_txt = '/content/dataset/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat ./data.yaml

Created config file at /content/data.yaml

File contents:

path: /content/original_training_dataset
train: train/images
val: validation/images
test: test/images
nc: 9
names:
- Bemisia tabaci (adult)
- Ceratitis capitata (adult)
- Dione juno (adult)
- Dione juno (larva)
- Ligyrus gibbosus (adult)
- Liriomyza huidobrensis (adult)
- Myzus persicae (nymph)
- Spodoptera frugiperda (adult)
- Spodoptera frugiperda (larva)


## **Evaluate the model (viewing results)**

In [ ]:
matplotlib.rcParams['savefig.dpi'] = 500
matplotlib.rcParams['figure.autolayout'] = True

from ultralytics import YOLO

# Load the best trained model
model = YOLO('/content/weights/best.pt')

# Evaluate on the test set
results = model.val(
    data='data.yaml',
    split='test',
    imgsz=640,
    conf=0.317,  # optimal threshold of the F1-Confidence curve
    iou=0.5,
    save_json=True,
    plots=True
)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,416,283 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1670.5±320.2 MB/s, size: 110.0 KB)
val: Scanning /content/original_training_dataset/test/labels... 135 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 135/135 698.0it/s 0.2s
val: New cache created: /content/original_training_dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.8it/s 5.1s
                   all        135        368      0.924      0.877      0.917       0.78
Bemisia tabaci (adult)         19         41      0.964      0.854      0.928      0.703
Ceratitis capitata (adult)         11         11      0.945          1      0.995      0.943
    Dione juno (adult)         13         13      0.929          1      0.995      0.925
    Dione juno (larva)         12         7

## **Export results**

In [ ]:
classes = results.names
ap_class_index = results.box.ap_class_index.tolist()
precision = results.box.p.tolist()
recall = results.box.r.tolist()
map50 = results.box.ap50.tolist()
map5095 = results.box.ap.tolist()

rows = [{
    'Class': 'all',
    'Precision': results.box.mp,
    'Recall': results.box.mr,
    'mAP@50': results.box.map50,
    'mAP@50-95': results.box.map,
}]

for i, idx in enumerate(ap_class_index):
    rows.append({
        'Class': classes[idx],
        'Precision': precision[i],
        'Recall': recall[i],
        'mAP@50': map50[i],
        'mAP@50-95': map5095[i],
    })

rows.append({'Class': '', 'Precision': '', 'Recall': '', 'mAP@50': '', 'mAP@50-95': ''})
rows.append({'Class': '', 'Precision': 'preprocess (ms)', 'Recall': 'inference (ms)', 'mAP@50': 'postprocess (ms)', 'mAP@50-95': ''})
rows.append({
    'Class': 'Speed per image',
    'Precision': results.speed['preprocess'],
    'Recall': results.speed['inference'],
    'mAP@50': results.speed['postprocess'],
    'mAP@50-95': '',
})

df = pd.DataFrame(rows)
df.to_csv('/content/runs/detect/val/results.csv', index=False)
print("Saved in results.csv")

shutil.make_archive('/content/results', 'zip', '/content/runs/detect/val')
files.download('/content/results.zip')

Saved in results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>